## Imports & src definitions

All `src/` logic (UMF, User, create_dataloader, decentralized_train_n_epochs, etc.)
is inlined here so the notebook is fully self-contained.

In [94]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.optim import SGD
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import networkx as nx
import time
import math
import copy
import os

torch.manual_seed(0)
np.random.seed(0)

In [96]:
from os import chdir
from pathlib import Path

new_path = Path("/Users/haowen/Documents/Decentral RS/fed-learning-main")

if new_path.exists():
    os.chdir(new_path)
    print(f"Working directory changed to: {Path.cwd()}")
else:
    print("Path does not exist.")


Working directory changed to: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [98]:
from src.models.MatrixFactorization import MF, UMF
from src.graphs import random_k_out_graph, create_graph
from src.users import User
from src.training.decentralized import (decentralized_train_n_epochs,
                                        decentralized_validate_loop)
from src.data_utils import create_batched_dataloaders, create_dataloader

In [100]:
#Make data sample iterable
from torch.utils.data import Dataset, DataLoader, TensorDataset, Sampler
import torch
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import SGD

from collections import Counter
import networkx as nx
from networkx.generators.classic import empty_graph
from networkx.utils import discrete_sequence, py_random_state, weighted_choice

import seaborn as sns
from sklearn.utils import shuffle

## Parameter Setting

In [103]:
para_vec= {
  "scalefree_userprop" : [0.006721468985407216, 0.3793755748581348, 0.7023494584199832],
  "scalefree_urs" : [0.00797255113179729, 0.7291631699209506, 0.7649689575684868],
  "scalefree_rs" : [0.043245636749499355, 0.24293301237845355, 0.6590721600407826],
  "scalefree_oaat" : [0.014505446034196021, 0.1281494707675557, 0.3063931184178566],
    
  "cycle_userprop" : [0.03448020025507248, 0.1530360406099725, 0.3265046312442892],
  "cycle_urs" : [0.015085184891905544, 0.32597756888723617, 0.9165691479123227],
  "cycle_rs" : [0.04518354114581989, 0.07432773840871296, 0.5104116722654509],
  "cycle_oaat" : [0.006051947990064438, 0.407449910177748, 0.6941867781038726],
    
  "random2_userprop" : [0.05973335259492166, 0.20270185084925238, 0.1],
  "random2_urs" : [0.03871364416669273, 0.14214480688557163, 0.4403378739685112],
  "random2_rs" : [0.03871364416669273, 0.14214480688557163, 0.01],
  "random2_oaat" : [0.012098247288774554, 0.051267232285266244, 0.5034632200402083],

  "random5_userprop" : [0.01214468819649195, 0.16071055871166323, 0.8930612583507401],
  "random5_urs" : [0.04664261576162963, 0.2261414992421005, 0.3645222958218734],
  "random5_rs" : [0.01864856189846265, 0.07043500222618476, 0.850748837624225],
  "random5_oaat" : [0.004358629931177893, 0.27784542450084454, 0.41295161556157467]
    
}

#lr = temp_para[0]
#weight_decay = temp_para[1]
#mom = temp_para[2]

In [105]:
# ── Hyperparameters — loaded from para_vec ────────────────────────────────────
# Set GRAPH_KEY to select the topology + sharing scheme.
# After running the tuning cell above, para_vec is updated automatically.
GRAPH_KEY = "random5_rs"   # change to match your current experiment

temp_para    = para_vec[GRAPH_KEY]
lr           = temp_para[0]
weight_decay = temp_para[1]
mom          = temp_para[2]

model_type        = "umf"
val_loader_type   = "rs"
train_loader_type = "rs"
userprop          = None
n_factors         = 30
sparse            = False
batch_size        = 10
graph_seed        = 1
n_epochs          = 50
SEED              = 42

print(f"Parameters for '{GRAPH_KEY}':")
print(f"  lr={lr:.6f}  weight_decay={weight_decay:.6f}  momentum={mom:.6f}")


Parameters for 'random5_rs':
  lr=0.018649  weight_decay=0.070435  momentum=0.850749


## Main

In [108]:
# ── Load sampled H&M dataset from cache ───────────────────────────────────────
# Set these to match what was used when sampling was run.
TARGET_USERS           = 1000
MIN_TRAIN_INTERACTIONS = 30
SAMPLED_DATA_DIR       = r"/Users/haowen/Documents/Decentral RS/rebuttal/code/hm_sampled"

cache_tag  = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s42"
train_path = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path   = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path  = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path  = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

assert all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]), (
    f"Cached files not found in '{SAMPLED_DATA_DIR}/'.\n"
    "Run the hm_foaf_experiment_sampled.ipynb preprocessing section first "
    "to generate the cache."
)

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)
meta_df  = pd.read_csv(meta_path)

n_users = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
n_items = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])

print(f"Loaded from cache: {cache_tag}")
print(f"Total User: {n_users}")
print(f"Total Item: {n_items}")
train_df.head()

Loaded from cache: u1000_m30_s42
Total User: 820
Total Item: 7290


,customer_id,product_code,bought
0,1,83,1
1,214,248,2
2,39,1466,1
3,406,2880,1
4,700,2293,1


In [110]:
# ── DataLoaders ───────────────────────────────────────────────────────────────
train_data_loader = create_dataloader(
    df=train_df, dl_type=train_loader_type, batch_size=batch_size, p=userprop
)
val_data_loader  = create_dataloader(df=val_df,  dl_type=val_loader_type)
test_data_loader = create_dataloader(df=test_df, dl_type=val_loader_type)

print(f"Train batches: {len(train_data_loader)} | "
      f"Val batches: {len(val_data_loader)} | "
      f"Test batches: {len(test_data_loader)}")

Train batches: 36112 | Val batches: 9029 | Test batches: 13717


## Parameter Tuning -- Grid Search

Run **after** cells 9-10 (data load + dataloaders). Calls `decentralized_train_n_epochs` for each combination in `LR_GRID x WEIGHT_DECAY_GRID x MOMENTUM_GRID` (27 combos by default), selects the combo with the lowest best validation RMSE, and writes it into `para_vec[TUNE_KEY]`. Re-run cell 7 then cells 13-16 to use the tuned parameters.


In [113]:
# ══════════════════════════════════════════════════════════════════════════════
# Parameter Tuning -- Grid Search
#
# Uses decentralized_train_n_epochs (same as the main training cell) on each
# (lr, weight_decay, momentum) combination. Selects the combo with the lowest
# best validation RMSE and writes it into para_vec[TUNE_KEY].
#
# Run AFTER cells 9-10 (data load + dataloaders).
# ══════════════════════════════════════════════════════════════════════════════
import itertools

# ── Grid ─────────────────────────────────────────────────────────────────────
TUNE_KEY      = "random5_rs"   # para_vec key to update
TUNE_EPOCHS   = 15             # max epochs per combo (short for speed)
TUNE_K        = 5              # graph k used during tuning

LR_GRID           = [0.005, 0.01,  0.05]
WEIGHT_DECAY_GRID = [0.05,  0.1,   0.3 ]
MOMENTUM_GRID     = [0.001, 0.5,   0.9 ]

param_grid = list(itertools.product(LR_GRID, WEIGHT_DECAY_GRID, MOMENTUM_GRID))
tune_graph = random_k_out_graph(n=n_users, k=TUNE_K, seed=graph_seed)

print(f"Grid search: {len(param_grid)} combinations x up to {TUNE_EPOCHS} epochs")
print(f"{'#':>3}  {'lr':>8}  {'wd':>8}  {'mom':>6}  {'best val RMSE':>14}")
print("-" * 48)

# ── Search ───────────────────────────────────────────────────────────────────
grid_results = []

for k, (lr_g, wd_g, mom_g) in enumerate(param_grid, 1):
    torch.manual_seed(0)

    # Fresh user models for this combo
    users_g = {}
    for i in range(n_users):
        m = UMF(n_items, n_factors=n_factors, sparse=sparse)
        users_g[i] = User(
            id=i, model=m,
            optimizer=SGD(m.parameters(),
                          lr=lr_g, weight_decay=wd_g, momentum=mom_g),
            model_name=model_type,
        )

    # Run with the same function as the main training cell
    _, val_losses_g, _, _ = decentralized_train_n_epochs(
        user_models=users_g,
        train_loader=train_data_loader,
        val_loader=val_data_loader,
        epochs=TUNE_EPOCHS,
        graph=tune_graph,
    )

    best_val = min(val_losses_g)
    grid_results.append((best_val, lr_g, wd_g, mom_g))

    marker = " <--" if best_val == min(r[0] for r in grid_results) else ""
    print(f"{k:>3}  {lr_g:>8.4f}  {wd_g:>8.4f}  {mom_g:>6.3f}  "
          f"{best_val:>14.4f}{marker}")

# ── Best combo ───────────────────────────────────────────────────────────────
best_val, best_lr, best_wd, best_mom = min(grid_results, key=lambda x: x[0])

print(f"\nBest val RMSE : {best_val:.4f}")
print(f"  lr           = {best_lr}")
print(f"  weight_decay = {best_wd}")
print(f"  momentum     = {best_mom}")

# ── Update para_vec ───────────────────────────────────────────────────────────
para_vec[TUNE_KEY] = [best_lr, best_wd, best_mom]
print(f"\npara_vec['{TUNE_KEY}'] updated. Re-run cell 7 then cells 13-16.")


Grid search: 27 combinations x up to 15 epochs
  #        lr        wd     mom   best val RMSE
------------------------------------------------


  0%|          | 0/36112 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 3.9816 | Validation Loss: 3.3823 | Time Elapsed: 38.409907 sec |Commute: 180392 | Commute 
Cost: 8160950880

  0%|          | 0/36112 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 2.0949 | Validation Loss: 2.8713 | Time Elapsed: 40.524212 sec |Commute: 180392 | Commute 
Cost: 8160950880

...

Early stopping.

Total time elapsed: 275.8225069581531

 20    0.0500    0.0500   0.500          0.4979


  0%|          | 0/36112 [00:00<?, ?it/s]

ValueError: NaN loss at step 5871, user 589

In [120]:
# ── Initialise one UMF model per user ─────────────────────────────────────────
users = {}
for i in tqdm(range(n_users)):
    user_model = UMF(n_items, n_factors=n_factors, sparse=sparse)
    users[i] = User(
        id=i,
        model=user_model,
        optimizer=SGD(user_model.parameters(),
                      lr=0.0100, weight_decay=0.3000,  momentum=0.001),
        model_name=model_type,
    )


  0%|          | 0/820 [00:00<?, ?it/s]

In [121]:
# ── Build graph ───────────────────────────────────────────────────────────────
graph = random_k_out_graph(n=n_users, k=5, seed=graph_seed)

In [123]:
# ── Train ─────────────────────────────────────────────────────────────────────
torch.manual_seed(0)
train_losses, val_losses, time_per_epoch, commutes = decentralized_train_n_epochs(
    user_models=users,
    train_loader=train_data_loader,
    val_loader=val_data_loader,
    epochs=n_epochs,
    graph=graph,
)

  0%|          | 0/36112 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2.8539 | Validation Loss: 1.7337 | Time Elapsed: 36.741899 sec |Commute: 180392 | Commute 
Cost: 8160950880

  0%|          | 0/36112 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 0.9818 | Validation Loss: 0.9773 | Time Elapsed: 40.113336 sec |Commute: 180392 | Commute 
Cost: 8160950880

  0%|          | 0/36112 [00:00<?, ?it/s]

...

Epoch 10 | Train Loss: 0.8040 | Validation Loss: 0.4583 | Time Elapsed: 34.020657 sec |Commute: 180392 | Commute 
Cost: 8160950880

  0%|          | 0/36112 [00:00<?, ?it/s]

Epoch 11 | Train Loss: 0.8175 | Validation Loss: 0.4565 | Time Elapsed: 43.838412 sec |Commute: 180392 | Commute 
Cost: 8160950880

Early stopping.

Total time elapsed: 398.66896550008096

In [124]:
# ── Test evaluation ───────────────────────────────────────────────────────────
test_loss = decentralized_validate_loop(users, test_data_loader)

In [125]:
test_loss

0.51966995

## Save Results